In [ ]:
import io
import os
import polars as pl
import urllib3
from dotenv import load_dotenv
import pandas as pd
import boto3
from botocore.client import Config

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
load_dotenv(".env")

s3 = boto3.client(
    "s3",
    endpoint_url          = os.getenv("MINIO_ENDPOINT"),
    aws_access_key_id     = os.getenv("MINIO_ACCESS_KEY"),
    aws_secret_access_key = os.getenv("MINIO_SECRET_KEY"),
    config                = Config(signature_version="s3v4"),
    region_name           = "us-east-1",
    verify                = False,
)

buf = io.BytesIO()
s3.download_fileobj("gold", "panel_admin/panel_complet.parquet", buf)
buf.seek(0)
base = pl.read_parquet(buf)

print(f"Base chargée : {base.shape[0]:,} lignes × {base.shape[1]} colonnes")
base.head(3)


/opt/anaconda3/lib/python3.12/site-packages/polars/_cpu_check.py:271: RuntimeWarning: Missing required CPU features.

The following required CPU features were not detected:
    avx, avx2, fma, bmi1, bmi2, lzcnt, movbe
Continuing to use this version of Polars on this processor will likely result in a crash.
Install `polars[rtcompat]` instead of `polars` to run Polars with better compatibility.

Hint: If you are on an Apple ARM machine (e.g. M1) this is likely due to running Python under Rosetta.
It is recommended to install a native version of Python that does not run under Rosetta x86-64 emulation.

If you believe this warning to be a false positive, you can set the `POLARS_SKIP_CPU_CHECK` environment variable to bypass this check.

  warnings.warn(


# 1. Vue d'ensemble du panel

Schéma, dtypes des 3 colonnes cibles, plage temporelle, nombre d'individus.

In [1]:
COLS_CIBLES = ["sexe", "date_naissance", "prise_service"]

print("Shape :", base.shape)
print()
print("Dtypes des colonnes cibles :")
for c in COLS_CIBLES:
    if c in base.columns:
        print(f"  {c:<20} {base.schema[c]}")
    else:
        print(f"  {c:<20} ABSENT")

print()
if "mois_annee" in base.columns:
    periodes = base["mois_annee"].drop_nulls().unique().sort()
    print(f"Périodes : {len(periodes)}  ·  min={periodes.min()}  max={periodes.max()}")

if "matricule" in base.columns:
    print(f"Matricules uniques : {base['matricule'].n_unique():,}")
if "cle_unique" in base.columns:
    print(f"cle_unique uniques : {base['cle_unique'].n_unique():,}")

base.select(COLS_CIBLES + [c for c in ["matricule", "cle_unique", "mois_annee"] if c in base.columns]).head(5)


NameError: name 'base' is not defined

# 2. Distribution des valeurs par colonne

## 2.1 `sexe` — encodage et fréquences

In [4]:
n = len(base)
n_null = base["sexe"].null_count()
print(f"sexe : {n_null:,} null / {n:,} obs ({100*n_null/n:.2f} %)")
print()
print("Top valeurs (lignes) :")
print(
    base.group_by("sexe")
        .agg(pl.len().alias("n_lignes"),
             pl.col("matricule").n_unique().alias("n_matricules"))
        .sort("n_lignes", descending=True)
)


sexe : 748 null / 32,196,625 obs (0.00 %)

Top valeurs (lignes) :


: 

## 2.2 `date_naissance` — formats détectés et parsabilité

In [1]:
def diagnose_date_col(df: pl.DataFrame, col: str) -> None:
    n = len(df)
    n_null = df[col].null_count()
    print(f"{col} : {n_null:,} null / {n:,} obs ({100*n_null/n:.2f} %)")
    print(f"  dtype : {df.schema[col]}")

    # Si déjà typée Date ou Datetime, on ne peut pas faire d'analyse de format string
    if df.schema[col] in (pl.Date, pl.Datetime):
        print(f"  Colonne déjà typée date — extrema : "
              f"min={df[col].min()}  max={df[col].max()}")
        return

    s = df[col].cast(pl.Utf8, strict=False).drop_nulls()
    print(f"  Valeurs distinctes (non-null) : {s.n_unique():,}")

    # Patterns de format approximatifs
    patterns = {
        "YYYY-MM-DD"  : r"^\d{4}-\d{2}-\d{2}$",
        "DD/MM/YYYY"  : r"^\d{2}/\d{2}/\d{4}$",
        "DD-MM-YYYY"  : r"^\d{2}-\d{2}-\d{4}$",
        "YYYY/MM/DD"  : r"^\d{4}/\d{2}/\d{2}$",
        "DD.MM.YYYY"  : r"^\d{2}\.\d{2}\.\d{4}$",
        "YYYY (4 chiffres seuls)": r"^\d{4}$",
        "datetime ISO": r"^\d{4}-\d{2}-\d{2}[T ]\d{2}:\d{2}",
    }
    print("  Détection patterns :")
    total_match = 0
    for nom, regex in patterns.items():
        n_match = s.str.contains(regex).sum()
        total_match += n_match
        if n_match > 0:
            print(f"    {nom:<28} {n_match:>10,}")
    n_autre = len(s) - total_match
    print(f"    {'autre / non reconnu':<28} {n_autre:>10,}")

    print("  Top 10 valeurs :")
    top = (df.group_by(col).agg(pl.len().alias("n")).sort("n", descending=True).head(10))
    for row in top.iter_rows(named=True):
        print(f"    {str(row[col])!r:<30}  {row['n']:>8,}")

    if n_autre > 0:
        print("  Échantillon de valeurs 'non reconnues' :")
        any_match = None
        for regex in patterns.values():
            m = s.str.contains(regex)
            any_match = m if any_match is None else (any_match | m)
        non_reco = s.filter(~any_match).unique().head(15).to_list()
        for v in non_reco:
            print(f"    {v!r}")


diagnose_date_col(base, "date_naissance")


NameError: name 'pl' is not defined

## 2.3 `prise_service` — formats détectés et parsabilité

In [ ]:
diagnose_date_col(base, "prise_service")

# 3. Cohérence intra-individu

Pour un même `matricule`, la valeur devrait être stable sur toutes les périodes.
On compte les valeurs distinctes (non null) par matricule pour chaque colonne cible.

In [ ]:
def coherence_intra_matricule(df: pl.DataFrame, col: str) -> pl.DataFrame:
    """Pour chaque matricule, compte les valeurs distinctes (non null) de col."""
    return (
        df.group_by("matricule")
          .agg([
              pl.col(col).drop_nulls().n_unique().alias("n_valeurs_distinctes"),
              pl.col(col).is_null().sum().alias("n_lignes_nulles"),
              pl.len().alias("n_lignes_totales"),
          ])
    )


resume_coherence = []
for col in COLS_CIBLES:
    coh = coherence_intra_matricule(base, col)
    n_matricules     = len(coh)
    n_zero_obs       = (coh["n_valeurs_distinctes"] == 0).sum()  # uniquement nulls
    n_une_valeur     = (coh["n_valeurs_distinctes"] == 1).sum()
    n_incoherent     = (coh["n_valeurs_distinctes"] > 1).sum()

    print(f"--- {col} ---")
    print(f"  matricules total           : {n_matricules:>8,}")
    print(f"  matricules sans aucune obs : {n_zero_obs:>8,}  ({100*n_zero_obs/n_matricules:.2f} %)")
    print(f"  matricules avec 1 valeur   : {n_une_valeur:>8,}  ({100*n_une_valeur/n_matricules:.2f} %)")
    print(f"  matricules incohérents (>1): {n_incoherent:>8,}  ({100*n_incoherent/n_matricules:.2f} %)")

    if n_incoherent > 0:
        repart = (
            coh.filter(pl.col("n_valeurs_distinctes") > 1)
               .group_by("n_valeurs_distinctes")
               .agg(pl.len().alias("n_matricules"))
               .sort("n_valeurs_distinctes")
        )
        print("  Répartition (n_valeurs_distinctes > 1) :")
        for row in repart.iter_rows(named=True):
            print(f"    {row['n_valeurs_distinctes']:>3} valeurs distinctes : {row['n_matricules']:>6,} matricules")
    print()

    resume_coherence.append({
        "col": col,
        "n_matricules_total": n_matricules,
        "n_matricules_aucune_obs": n_zero_obs,
        "n_matricules_1_valeur": n_une_valeur,
        "n_matricules_incoherent": n_incoherent,
    })

resume_coherence = pl.DataFrame(resume_coherence)
resume_coherence


## 3.1 Échantillon de matricules incohérents

Pour chaque colonne, on affiche jusqu'à 15 matricules qui ont >1 valeurs distinctes,
avec la liste des valeurs observées et leurs effectifs.

In [ ]:
def echantillon_incoherents(df: pl.DataFrame, col: str, k: int = 15) -> pl.DataFrame:
    coh = coherence_intra_matricule(df, col)
    matricules_pb = (
        coh.filter(pl.col("n_valeurs_distinctes") > 1)
           .head(k)["matricule"]
           .to_list()
    )
    if not matricules_pb:
        return pl.DataFrame()
    return (
        df.filter(pl.col("matricule").is_in(matricules_pb))
          .group_by(["matricule", col])
          .agg(pl.len().alias("n_lignes"))
          .sort(["matricule", "n_lignes"], descending=[False, True])
    )


for col in COLS_CIBLES:
    print(f"\n=== Échantillon incohérences : {col} ===")
    ech = echantillon_incoherents(base, col, k=10)
    if len(ech) == 0:
        print("  (aucun matricule incohérent)")
    else:
        with pl.Config(tbl_rows=80, fmt_str_lengths=60):
            print(ech)


## 3.2 Croisement : matricules avec multi-incohérences

Combien de matricules sont incohérents simultanément sur 2 ou 3 colonnes ?

In [ ]:
flags = (
    base.group_by("matricule")
        .agg([
            (pl.col(c).drop_nulls().n_unique() > 1).alias(f"incoh_{c}")
            for c in COLS_CIBLES
        ])
        .with_columns(
            (pl.col(f"incoh_{COLS_CIBLES[0]}").cast(pl.Int8)
             + pl.col(f"incoh_{COLS_CIBLES[1]}").cast(pl.Int8)
             + pl.col(f"incoh_{COLS_CIBLES[2]}").cast(pl.Int8)
            ).alias("nb_cols_incoherentes")
        )
)

print("Distribution du nombre de colonnes incohérentes par matricule :")
print(
    flags.group_by("nb_cols_incoherentes")
         .agg(pl.len().alias("n_matricules"))
         .sort("nb_cols_incoherentes")
)

print("\nMatricules incohérents sur les 3 colonnes simultanément :",
      (flags["nb_cols_incoherentes"] == 3).sum())


# 4. Cohérence calendaire

Parsing des dates en `pl.Date` (best-effort, multi-format) puis vérifications :

- âge à `mois_annee` (entre 16 et ~80 ans)
- `prise_service` ne devrait pas être **postérieure** à `mois_annee`
- âge à la prise de service plausible (16-65 ans)
- ordre `date_naissance < prise_service < date_retraite`

In [ ]:
def parse_date_multi(col_expr: pl.Expr) -> pl.Expr:
    """Tente plusieurs formats. Renvoie pl.Date ou null si non parsable.
    Si la colonne est déjà Date/Datetime, conversion directe."""
    return (
        pl.coalesce([
            col_expr.cast(pl.Utf8, strict=False).str.to_date("%Y-%m-%d", strict=False),
            col_expr.cast(pl.Utf8, strict=False).str.to_date("%d/%m/%Y", strict=False),
            col_expr.cast(pl.Utf8, strict=False).str.to_date("%d-%m-%Y", strict=False),
            col_expr.cast(pl.Utf8, strict=False).str.to_date("%Y/%m/%d", strict=False),
            col_expr.cast(pl.Utf8, strict=False).str.to_date("%d.%m.%Y", strict=False),
            col_expr.cast(pl.Utf8, strict=False).str.to_datetime(strict=False).cast(pl.Date),
        ])
    )


# mois_annee est en MMYYYY (string) — on le convertit en première date du mois
def parse_mois_annee(col_expr: pl.Expr) -> pl.Expr:
    s = col_expr.cast(pl.Utf8, strict=False).str.replace_all(r"[_/.\-]", "")
    return pl.coalesce([
        # MMYYYY
        ("01-" + s.str.slice(2, 4) + "-" + s.str.slice(0, 2))
            .str.to_date("%d-%Y-%m", strict=False),
        # YYYYMM
        ("01-" + s.str.slice(0, 4) + "-" + s.str.slice(4, 2))
            .str.to_date("%d-%Y-%m", strict=False),
    ])


base_p = base.with_columns([
    parse_date_multi(pl.col("date_naissance")).alias("dn_dt"),
    parse_date_multi(pl.col("prise_service")).alias("ps_dt"),
    parse_mois_annee(pl.col("mois_annee")).alias("ma_dt"),
])

if "date_retraite" in base.columns:
    base_p = base_p.with_columns(parse_date_multi(pl.col("date_retraite")).alias("dr_dt"))

# Statistiques de parsing
n = len(base_p)
for src, dst in [("date_naissance", "dn_dt"),
                 ("prise_service",  "ps_dt"),
                 ("mois_annee",     "ma_dt")]:
    n_in_null  = base_p[src].null_count() if base_p[src].dtype != pl.Date else base_p[src].null_count()
    n_out_null = base_p[dst].null_count()
    n_parsed   = n - n_out_null
    print(f"{src:<16} → parsé : {n_parsed:>10,} / {n:,}  "
          f"({100*n_parsed/n:5.2f} %)  ·  in_null={n_in_null:,}  out_null={n_out_null:,}")


## 4.1 Bornes des dates parsées

In [ ]:
for col in ["dn_dt", "ps_dt", "ma_dt"] + (["dr_dt"] if "dr_dt" in base_p.columns else []):
    s = base_p[col].drop_nulls()
    if len(s) == 0:
        print(f"{col} : vide")
        continue
    print(f"{col:<8} min={s.min()}  max={s.max()}  "
          f"q05={s.quantile(0.05)}  q50={s.quantile(0.50)}  q95={s.quantile(0.95)}")


## 4.2 Anomalies calendaires

- **age_obs** = âge à la période (`mois_annee`) — devrait être ∈ [16, 80]
- **prise_service > mois_annee** : agent observé avant sa prise de service ?
- **age_prise_service** : âge à la prise de service — devrait être ∈ [16, 65]
- **date_naissance >= prise_service** : prise de service avant la naissance ?
- **prise_service >= date_retraite** (si dispo) : retraite avant la prise de service ?

In [ ]:
DAYS_PER_YEAR = 365.25

base_a = base_p.with_columns([
    ((pl.col("ma_dt") - pl.col("dn_dt")).dt.total_days() / DAYS_PER_YEAR)
        .alias("age_obs"),
    ((pl.col("ps_dt") - pl.col("dn_dt")).dt.total_days() / DAYS_PER_YEAR)
        .alias("age_prise_service"),
])

n = len(base_a)

def pct(n_a):
    return f"{n_a:>10,}  ({100*n_a/n:5.2f} %)"

print(f"Total observations : {n:,}")
print()

# Âge à la période
mask = base_a["age_obs"].is_not_null()
n_age_lt_16  = ((base_a["age_obs"] < 16)  & mask).sum()
n_age_gt_80  = ((base_a["age_obs"] > 80)  & mask).sum()
n_age_neg    = ((base_a["age_obs"] < 0)   & mask).sum()
print(f"age_obs < 0   (naissance après mois_annee) : {pct(n_age_neg)}")
print(f"age_obs < 16  (mineur)                     : {pct(n_age_lt_16)}")
print(f"age_obs > 80  (très âgé)                   : {pct(n_age_gt_80)}")
print()

# Observation avant prise de service
mask = base_a["ps_dt"].is_not_null() & base_a["ma_dt"].is_not_null()
n_avant_ps = ((base_a["ps_dt"] > base_a["ma_dt"]) & mask).sum()
print(f"prise_service > mois_annee (obs. avant prise svc) : {pct(n_avant_ps)}")

# Âge à la prise de service
mask = base_a["age_prise_service"].is_not_null()
n_aps_lt_16 = ((base_a["age_prise_service"] < 16) & mask).sum()
n_aps_gt_65 = ((base_a["age_prise_service"] > 65) & mask).sum()
n_aps_neg   = ((base_a["age_prise_service"] < 0)  & mask).sum()
print(f"age_prise_service < 0  (prise svc avant naiss.)   : {pct(n_aps_neg)}")
print(f"age_prise_service < 16                            : {pct(n_aps_lt_16)}")
print(f"age_prise_service > 65                            : {pct(n_aps_gt_65)}")
print()

if "dr_dt" in base_a.columns:
    mask = base_a["dr_dt"].is_not_null() & base_a["ps_dt"].is_not_null()
    n_dr_lt_ps = ((base_a["dr_dt"] < base_a["ps_dt"]) & mask).sum()
    print(f"date_retraite < prise_service                   : {pct(n_dr_lt_ps)}")


# 5. Synthèse

Tableau récapitulatif pour décider la stratégie de nettoyage de chaque colonne.

In [ ]:
n_obs       = len(base)
n_matricules = base["matricule"].n_unique()

synthese = []
for col, dt_parsed in [("sexe", None),
                       ("date_naissance", "dn_dt"),
                       ("prise_service",  "ps_dt")]:
    n_null   = base[col].null_count()
    pct_null = round(100 * n_null / n_obs, 2)

    coh = coherence_intra_matricule(base, col)
    n_incoh = (coh["n_valeurs_distinctes"] > 1).sum()
    pct_incoh = round(100 * n_incoh / n_matricules, 2)

    if dt_parsed is None:
        pct_parsable = None
    else:
        n_parsable = n_obs - base_p[dt_parsed].null_count()
        # On compare au nb d'obs non-nulles initiales (parsable rapporté aux non-nulls)
        n_non_null = n_obs - n_null
        pct_parsable = round(100 * n_parsable / n_non_null, 2) if n_non_null else None

    synthese.append({
        "colonne": col,
        "n_obs": n_obs,
        "%_null": pct_null,
        "%_parsable_si_non_null": pct_parsable,
        "matricules_incoherents": n_incoh,
        "%_matricules_incoherents": pct_incoh,
    })

synthese_df = pl.DataFrame(synthese)
print("SYNTHÈSE — état des 3 colonnes au panel `panel_complet.parquet`")
print(synthese_df)


---

**Prochaine étape** — sur la base de cette exploration, décider :

1. **`sexe`** — quelle stratégie de réconciliation par matricule ? (mode majoritaire, première valeur, etc.) Faut-il aussi standardiser l'encodage en `Homme`/`Femme` (mapping déjà disponible dans `07_calcul_indicateur.py` lignes 873-900) ?
2. **`date_naissance`** — convertir en `pl.Date`, écarter les dates aberrantes (bornes à fixer), puis réconcilier par matricule.
3. **`prise_service`** — convertir en `pl.Date`, choisir entre min, mode ou première valeur observée par matricule.
4. Décider du livrable : nouveau fichier `panel_complet_clean.parquet` à côté de l'original, ou réécriture en place.